# FD003 — faithful Peng reproduction, factor ablation, and two search algorithms

## The benchmark value was wrong

Read from the paper (Peng et al., *Sensors* 21(2):418, 2021):

| | FD001 | FD003 |
|---|---|---|
| Table 5 — RMSE **training** values | 4.87 | **7.56** |
| Table 6 — RMSE **predicted** (test) | 11.17 | **9.99** |

Our benchmark table has been citing **7.56, their training-set RMSE**, as a published test result.
The correct comparison value is **9.99**. This needs fixing in the manuscript, and it explains why
no faithful implementation could approach it.

## What the paper actually specifies

| | Peng | our previous runs |
|---|---|---|
| features | **25** = 21 sensors + 3 settings + running-cycle index | 14 selected sensors |
| window | **50** | 30 |
| normalisation | **Min-Max to [0,1]** | per-condition StandardScaler |
| RUL cap | 125 (piecewise linear) | 125 |
| training | 60 epochs, batch 512, dropout 0.2, Adam, early stopping | 50 epochs, batch 32 |

Architecture, exactly as written in section 3.2: the CNN path is
`Conv1D(128,3) - MaxPool(2,same,s=2) - BN`, then the same with 64 filters, then 32; the LSTM path
is `LSTM(128) - LSTM(64) - LSTM(32)`, all `return_sequences=True`; the two are concatenated and
passed to `Conv1D(256,3) - MaxPool - Flatten - Dense(128) - Dense(32) - Dense(1)`.

Both paths end at **32 channels**, which is what makes the concatenation well-defined: it is along
the **time axis**. With window 50 the CNN path is 50 -> 25 -> 13 -> 7 steps and the LSTM path stays
at 50, so the fusion is (57, 32).

## Structure

**Part A** reproduces Peng exactly and compares against 9.99.
**Part B** is a 2x2: {Peng architecture, our architecture} x {Peng preprocessing, our
preprocessing}. This separates how much of their advantage is the architecture and how much is
the 25-feature, window-50, Min-Max pipeline — which is what tells us what to adopt.
**Part C** runs a larger Bayesian search and a particle-swarm search on the winner, both anchored
so neither can return worse than its starting point.
**Part D** rebuilds the benchmark table with the corrected value.

Run from the folder containing `train_FD003.txt`.

In [1]:
import os, json, gc, time, random, math
os.environ['PYTHONHASHSEED']='42'
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Concatenate, Flatten, BatchNormalization, Softmax,
                                     Multiply, Lambda, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt

def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)

def gpu_preflight():
    if not tf.config.list_physical_devices('GPU'): print('no GPU - CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s) - CPU fallback. Close other kernels holding the GPU.'
              %type(e).__name__); K.clear_session(); return False
USE_GPU=gpu_preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

GPU preflight passed
TF 2.10.0 | device /GPU:0


In [2]:
SEED=42; RUL_CAP=125
SEEDS_MAIN, SEEDS_FINAL = 5, 5
SEARCH_SEEDS, COMPARE_SEEDS = 1, 3
BO_TRIALS   = 20
PSO_PART, PSO_ITER = 8, 4
EPOCHS = 60          # ONE budget for search, comparison and final
set_seeds(SEED)

idx=['unit_nr','time_cycles']; setn=['setting_1','setting_2','setting_3']
sn=[f's_{i}' for i in range(1,22)]; cols=idx+setn+sn
train=pd.read_csv('train_FD003.txt',sep=r'\s+',header=None,names=cols)
test =pd.read_csv('test_FD003.txt', sep=r'\s+',header=None,names=cols)
y_test=pd.read_csv('RUL_FD003.txt',sep=r'\s+',header=None,names=['RUL'])
OURS_KEEP=['s_2','s_3','s_4','s_7','s_8','s_9','s_11','s_12','s_13','s_14','s_15','s_17','s_20','s_21']

# ---- engine split decided FIRST, so scalers never see validation engines --------
_g=GroupShuffleSplit(n_splits=1,train_size=0.8,random_state=SEED)
_ti,_vi=next(_g.split(train,groups=train.unit_nr))
TRAIN_UNITS=np.sort(train.iloc[_ti].unit_nr.unique())
VAL_UNITS  =np.sort(train.iloc[_vi].unit_nr.unique())
assert not set(TRAIN_UNITS)&set(VAL_UNITS)
print('engines: %d train / %d validation / %d test | overlap %d'
      %(len(TRAIN_UNITS),len(VAL_UNITS),test.unit_nr.nunique(),
        len(set(TRAIN_UNITS)&set(VAL_UNITS))))

def add_rul(df):
    r=df.copy(); mx=r.groupby('unit_nr')['time_cycles'].transform('max')
    r['RUL']=(mx-r['time_cycles']).clip(upper=RUL_CAP); return r

def prep_peng(use_cycle=True):
    """Peng: 21 sensors + 3 settings (+ running-cycle index), Min-Max to [0,1], window 50."""
    feats=setn+sn+(['time_cycles'] if use_cycle else [])
    tr=add_rul(train.copy()); te=test.copy()
    sc=MinMaxScaler().fit(tr[tr.unit_nr.isin(TRAIN_UNITS)][feats])   # train engines only
    tr[feats]=np.nan_to_num(sc.transform(tr[feats]))
    te[feats]=np.nan_to_num(sc.transform(te[feats]))
    return tr,te,feats,50

def prep_ours():
    """Ours: 14 selected sensors, per-condition StandardScaler, window 30."""
    feats=OURS_KEEP
    def cond(df):
        r=df.copy(); r['setting_1']=r['setting_1'].round(); r['setting_2']=r['setting_2'].round(2)
        r['op_cond']=r['setting_1'].astype(str)+'_'+r['setting_2'].astype(str)+'_'+r['setting_3'].astype(str)
        return r
    tr=cond(train.copy()); te=cond(test.copy())
    for d in (tr,te): d[feats]=d[feats].astype(np.float64)
    fit_rows=tr[tr.unit_nr.isin(TRAIN_UNITS)]                        # train engines only
    cs={str(c):StandardScaler().fit(g[feats]) for c,g in fit_rows.groupby('op_cond')}
    gs=StandardScaler().fit(fit_rows[feats])
    def ap(df):
        r=df.copy()
        for c,ix in r.groupby('op_cond').groups.items():
            r.loc[ix,feats]=cs.get(str(c),gs).transform(r.loc[ix,feats])
        return r
    return add_rul(ap(tr)),ap(te),feats,30

def build_arrays(tr,te,feats,W):
    def win(df,us):
        X,y=[],[]
        for u in us:
            d=df[df.unit_nr==u]; A=d[feats].to_numpy(np.float32); L=d['RUL'].to_numpy(np.float32)
            for e in range(W,len(d)+1): X.append(A[e-W:e]); y.append(L[e-1])
        return np.asarray(X,np.float32),np.asarray(y,np.float32).reshape(-1,1)
    Xtr,ytr=win(tr,TRAIN_UNITS); Xva,yva=win(tr,VAL_UNITS)
    Xte=[]
    for u in te.unit_nr.unique():
        A=te[te.unit_nr==u][feats].to_numpy(np.float32)
        Xte.append(A[-W:] if len(A)>=W else np.vstack([np.repeat(A[:1],W-len(A),axis=0),A]))
    Xte=np.asarray(Xte,np.float32)
    yte=np.clip(y_test['RUL'].to_numpy(np.float32),0,RUL_CAP).reshape(-1,1)
    return dict(Xtr=Xtr,ytr=ytr,Xva=Xva,yva=yva,Xte=Xte,yte=yte,W=W,F=len(feats))

DATA={}
for nm,fn in (('peng',lambda: prep_peng(True)),
              ('peng_nocyc',lambda: prep_peng(False)),
              ('ours',prep_ours)):
    tr,te,f,W=fn(); DATA[nm]=build_arrays(tr,te,f,W); d=DATA[nm]
    print('%-11s window %2d | %2d features | train %s | val %s | test %s'
          %(nm,d['W'],d['F'],d['Xtr'].shape,d['Xva'].shape,d['Xte'].shape))
YTE=DATA['ours']['yte']
TRIVIAL=float(np.sqrt(mean_squared_error(YTE,np.full_like(YTE,DATA['ours']['ytr'].mean()))))
print('trivial baseline (predict training mean): %.2f'%TRIVIAL)

# ---- leakage assertions ---------------------------------------------------------
assert not set(TRAIN_UNITS)&set(VAL_UNITS), 'train/val engine overlap'
for nm,d in DATA.items():
    assert d['Xte'].shape[0]==100 and d['yte'].shape[0]==100, nm
print('leakage checks passed: engines disjoint, scalers fitted on training engines only,')
print('test used once at evaluation with labels from RUL_FD003.txt')

engines: 80 train / 20 validation / 100 test | overlap 0
peng        window 50 | 25 features | train (16092, 50, 25) | val (3728, 50, 25) | test (100, 50, 25)
peng_nocyc  window 50 | 24 features | train (16092, 50, 24) | val (3728, 50, 24) | test (100, 50, 24)
ours        window 30 | 14 features | train (17692, 30, 14) | val (4128, 30, 14) | test (100, 30, 14)
trivial baseline (predict training mean): 42.17
leakage checks passed: engines disjoint, scalers fitted on training engines only,
test used once at evaluation with labels from RUL_FD003.txt


## Architectures

In [3]:
def peng_model(shape,seed=SEED,f1=128,f2=64,f3=32,l1=128,l2_=64,l3=32,fuse=256,
               d1=128,d2=32,dropout=0.2,lr=1e-3,l2reg=0.0):
    """1-FCLCNN-LSTM exactly as specified in Peng et al. section 3.2."""
    kw=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
            kernel_regularizer=regularizers.l2(l2reg) if l2reg>0 else None)
    inp=Input(shape=shape)
    c=inp
    for f in (f1,f2,f3):                                  # 3 conv + pool + BN
        c=Conv1D(f,3,padding='same',activation='relu',**kw)(c)
        c=MaxPooling1D(2,padding='same',strides=2)(c)
        c=BatchNormalization()(c)
    if dropout>0: c=Dropout(dropout,seed=seed)(c)
    r=inp
    for u in (l1,l2_,l3):                                 # 3 LSTM, all return_sequences
        r=LSTM(u,return_sequences=True,**kw)(r)
    f=Concatenate(axis=1)([c,r])                          # both paths end at 32 channels
    f=Conv1D(fuse,3,padding='same',activation='relu',**kw)(f)
    f=MaxPooling1D(2,padding='same',strides=2)(f)
    f=Flatten()(f)
    f=Dense(d1,activation='relu',**kw)(f)
    if dropout>0: f=Dropout(dropout,seed=seed)(f)
    f=Dense(d2,activation='relu',**kw)(f)
    m=Model(inp,Dense(1,activation='linear',**kw)(f))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='mse'); return m

def ours_model(shape,seed=SEED,filters=128,lstm_units=128,dense=128,merged=256,
               dropout=0.2,l2reg=0.01,lr=1e-3):
    """Our parallel CNN-LSTM."""
    kw=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed))
    inp=Input(shape=shape)
    c=Conv1D(filters,3,padding='same',activation='relu',**kw)(inp)
    c=MaxPooling1D(2,padding='same')(c); c=Flatten()(c)
    c=Dense(dense,activation='relu',kernel_regularizer=regularizers.l2(l2reg),**kw)(c)
    c=Dropout(dropout,seed=seed)(c)
    r=LSTM(lstm_units,return_sequences=True,**kw)(inp)
    r=LSTM(lstm_units,activation='tanh',**kw)(r)
    r=Dense(dense,activation='relu',kernel_regularizer=regularizers.l2(l2reg),**kw)(r)
    r=Dropout(dropout,seed=seed)(r)
    h=Concatenate()([c,r])
    h=Dense(merged,activation='relu',kernel_regularizer=regularizers.l2(l2reg),**kw)(h)
    h=Dropout(dropout,seed=seed)(h)
    m=Model(inp,Dense(1,activation='linear',**kw)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='mse'); return m

ARCH={'peng':peng_model,'ours':ours_model}
BATCH={'peng':512,'ours':32}
for a in ARCH:
    for p in DATA:
        mm=ARCH[a]((DATA[p]['W'],DATA[p]['F']))
        print('%-5s arch on %-5s prep -> %s params'%(a,p,format(mm.count_params(),',')))
        K.clear_session()

peng  arch on peng  prep -> 1,161,505 params
peng  arch on peng_nocyc prep -> 1,160,609 params
peng  arch on ours  prep -> 758,433 params
ours  arch on peng  prep -> 712,449 params
ours  arch on peng_nocyc prep -> 711,553 params
ours  arch on ours  prep -> 538,753 params


## Parts A and B — faithful reproduction and the 2x2 factor ablation

In [4]:
def run(arch,prep,seed,epochs=None,**kw):
    d=DATA[prep]; ep=epochs or EPOCHS
    K.clear_session(); gc.collect(); set_seeds(seed)
    with tf.device(DEVICE):
        m=ARCH[arch]((d['W'],d['F']),seed=seed,**kw)
        m.fit(d['Xtr'],d['ytr'],validation_data=(d['Xva'],d['yva']),epochs=ep,
              batch_size=BATCH[arch],verbose=0,
              callbacks=[EarlyStopping('val_loss',patience=15,restore_best_weights=True),
                         ReduceLROnPlateau('val_loss',factor=0.5,patience=5,min_lr=1e-5)])
        v=float(np.sqrt(mean_squared_error(d['yva'],m.predict(d['Xva'],batch_size=512,verbose=0))))
        t=float(np.sqrt(mean_squared_error(d['yte'],m.predict(d['Xte'],batch_size=512,verbose=0))))
    return v,t,m.count_params()

grid=[]; t0=time.time()
for arch in ('peng','ours'):
    for prep in ('peng','peng_nocyc','ours'):
        vs,ts,prm=[],[],None
        for s in range(SEEDS_MAIN):
            v,t,prm=run(arch,prep,SEED+s); vs.append(v); ts.append(t)
        grid.append(dict(arch=arch,prep=prep,params=int(prm),
                         val_mean=float(np.mean(vs)),
                         test_mean=float(np.mean(ts)),test_std=float(np.std(ts,ddof=1)),
                         test_best=float(np.min(ts)),test_all=[round(x,3) for x in ts]))
        print('%-5s arch + %-10s prep : test %.2f +/- %.2f | best %.2f | params %s'
              %(arch,prep,np.mean(ts),np.std(ts,ddof=1),np.min(ts),format(prm,',')))
G=pd.DataFrame(grid); pd.set_option('display.width',200)
print('\n'+G[['arch','prep','val_mean','test_mean','test_std','test_best','params']].to_string(index=False))
print('\nPeng published (test, Table 6): 9.99')
pp=float(G[(G.arch=='peng')&(G.prep=='peng')]['test_mean'].iloc[0])
print('our faithful reproduction      : %.2f  (%+.2f vs published)'%(pp,pp-9.99))
print('\n--- factor effects (mean over the other factor) ---')
print('architecture: peng %.2f vs ours %.2f'%(G[G.arch=="peng"].test_mean.mean(),G[G.arch=="ours"].test_mean.mean()))
print('preprocessing: peng %.2f | peng_nocyc %.2f | ours %.2f'%(
    G[G.prep=="peng"].test_mean.mean(),G[G.prep=="peng_nocyc"].test_mean.mean(),
    G[G.prep=="ours"].test_mean.mean()))
print('cycle-index feature contributes: %+.2f RMSE'%(
    G[G.prep=="peng"].test_mean.mean()-G[G.prep=="peng_nocyc"].test_mean.mean()))
BEST=G.sort_values('val_mean').iloc[0]
BA,BP=BEST['arch'],BEST['prep']
print('\nbest on validation: %s arch + %s prep (test %.2f)'%(BA,BP,BEST['test_mean']))
print('total %.0f s'%(time.time()-t0))

peng  arch + peng       prep : test 18.19 +/- 1.18 | best 17.05 | params 1,161,505
peng  arch + peng_nocyc prep : test 19.15 +/- 0.72 | best 18.38 | params 1,160,609
peng  arch + ours       prep : test 16.61 +/- 0.64 | best 15.84 | params 758,433
ours  arch + peng       prep : test 11.12 +/- 0.42 | best 10.62 | params 712,449
ours  arch + peng_nocyc prep : test 13.08 +/- 0.28 | best 12.79 | params 711,553
ours  arch + ours       prep : test 13.05 +/- 0.54 | best 12.16 | params 538,753

arch       prep  val_mean  test_mean  test_std  test_best  params
peng       peng 15.104914  18.186909  1.183720  17.053881 1161505
peng peng_nocyc 17.159767  19.152184  0.716432  18.382538 1160609
peng       ours 16.286837  16.613657  0.643242  15.844887  758433
ours       peng  8.448338  11.124947  0.417414  10.624847  712449
ours peng_nocyc  9.765152  13.076280  0.278926  12.793465  711553
ours       ours 12.081767  13.046968  0.542939  12.159546  538753

Peng published (test, Table 6): 9.99
our faith

## Part C — larger Bayesian search and particle swarm

Both search the same space, both are scored on the same objective, and both are anchored: the
baseline configuration is evaluated on that objective first and kept unless a search beats it.

In [5]:
SPACE = ([('f1',32,192,'int'),('f2',16,128,'int'),('f3',16,96,'int'),
          ('l1',32,192,'int'),('l2_',16,128,'int'),('l3',16,96,'int'),
          ('fuse',64,384,'int'),('d1',32,256,'int'),('d2',16,128,'int'),
          ('dropout',0.0,0.5,'float'),('lr',1e-4,5e-3,'log'),('l2reg',1e-7,1e-2,'log')]
         if BA=='peng' else
         [('filters',32,256,'int'),('lstm_units',32,256,'int'),('dense',32,256,'int'),
          ('merged',64,384,'int'),('dropout',0.0,0.5,'float'),
          ('lr',1e-4,5e-3,'log'),('l2reg',1e-7,1e-1,'log')])
DEFAULTS = ({'f1':128,'f2':64,'f3':32,'l1':128,'l2_':64,'l3':32,'fuse':256,
             'd1':128,'d2':32,'dropout':0.2,'lr':1e-3,'l2reg':0.0} if BA=='peng' else
            {'filters':128,'lstm_units':128,'dense':128,'merged':256,
             'dropout':0.2,'lr':1e-3,'l2reg':0.01})
print('search space: %d hyperparameters on the %s architecture'%(len(SPACE),BA))
for n,lo,hi,k in SPACE: print('   %-11s %-6s %-10s default %s'%(n,k,'%s..%s'%(lo,hi),DEFAULTS[n]))

def decode(vec):
    kw={}
    for (n,lo,hi,k),x in zip(SPACE,vec):
        x=float(min(max(x,0.0),1.0))
        if k=='int':   kw[n]=int(round(lo+x*(hi-lo)))
        elif k=='log': kw[n]=float(math.exp(math.log(lo)+x*(math.log(hi)-math.log(lo))))
        else:          kw[n]=float(lo+x*(hi-lo))
    return kw

def encode(kw):
    """Inverse of decode - lets the baseline be placed directly into the swarm."""
    v=[]
    for n,lo,hi,k in SPACE:
        x=kw[n]
        if k=='log': x=max(x,lo); v.append((math.log(x)-math.log(lo))/(math.log(hi)-math.log(lo)))
        else:        v.append((x-lo)/(hi-lo))
    return np.clip(np.array(v,float),0,1)

BASE_VEC=encode(DEFAULTS)
assert all(abs(decode(BASE_VEC)[n]-DEFAULTS[n])<=max(1e-9,0.02*abs(DEFAULTS[n])+1)
           for n,_,_,_ in SPACE), 'encode/decode round-trip failed'
print('\nbaseline encodes/decodes cleanly -> it can be seeded into the swarm')

def objective(kw,n_seeds=None,tag=''):
    """Validation RMSE at the SAME budget used everywhere else."""
    n=n_seeds or SEARCH_SEEDS; sc=[]
    for s in range(n):
        try:
            v,_,_=run(BA,BP,SEED+s,epochs=EPOCHS,**kw)
            if not np.isfinite(v): v=1e3
        except Exception as e:
            print('     fail (%s)'%type(e).__name__); v=1e3
        sc.append(v)
    return float(np.mean(sc))

BASE_OBJ=objective({},n_seeds=SEARCH_SEEDS)
print('baseline objective at %d epochs, %d seed(s): %.3f'%(EPOCHS,SEARCH_SEEDS,BASE_OBJ))

search space: 7 hyperparameters on the ours architecture
   filters     int    32..256    default 128
   lstm_units  int    32..256    default 128
   dense       int    32..256    default 128
   merged      int    64..384    default 256
   dropout     float  0.0..0.5   default 0.2
   lr          log    0.0001..0.005 default 0.001
   l2reg       log    1e-07..0.1 default 0.01

baseline encodes/decodes cleanly -> it can be seeded into the swarm
baseline objective at 60 epochs, 1 seed(s): 8.255


In [6]:
# ---------------- Bayesian ----------------
OBJ='val_rmse'
def hp_kw(hp):
    kw={}
    for n,lo,hi,k in SPACE:
        if k=='int':   kw[n]=hp.Int(n,lo,hi,step=max(1,(hi-lo)//16))
        elif k=='log': kw[n]=hp.Float(n,lo,hi,sampling='log')
        else:          kw[n]=hp.Float(n,lo,hi,step=0.05)
    return kw
class AnchoredBO(kt.BayesianOptimization):
    def run_trial(self,trial,*a,**k):
        v=objective(hp_kw(trial.hyperparameters))
        print('   BO trial %s: %.3f (baseline %.3f)'%(trial.trial_id,v,BASE_OBJ)); return {OBJ:v}
t0=time.time()
tuner=AnchoredBO(hypermodel=lambda hp: ARCH[BA]((DATA[BP]['W'],DATA[BP]['F']),**hp_kw(hp)),
                 objective=kt.Objective(OBJ,direction='min'),max_trials=BO_TRIALS,seed=SEED,
                 directory='search_fd003',project_name='bo_%s_%s'%(BA,BP),overwrite=False)
tuner.search()
bo_hp=tuner.get_best_hyperparameters(1)[0]; BO_KW=hp_kw(bo_hp)
BO_OBJ=min(t.score for t in tuner.oracle.trials.values()
           if t.score is not None and np.isfinite(t.score))
print('Bayesian best objective %.3f in %.0f s'%(BO_OBJ,time.time()-t0))

Reloading Tuner from search_fd003\bo_ours_peng\tuner0.json
Bayesian best objective 8.628 in 0 s


In [7]:
# ---------------- Particle swarm, seeded with the baseline ----------------
def pso(fitness,dim,x0=None,n_part=PSO_PART,iters=PSO_ITER,w=0.72,c1=1.5,c2=1.5,seed=SEED):
    rng=np.random.RandomState(seed)
    X=rng.rand(n_part,dim)
    if x0 is not None: X[0]=x0            # the baseline IS particle 0
    V=rng.uniform(-0.1,0.1,(n_part,dim))
    P=X.copy(); Pf=np.array([fitness(x) for x in X])
    g=int(np.argmin(Pf)); G=P[g].copy(); Gf=float(Pf[g])
    print('  PSO init best %.3f (particle 0 = baseline: %.3f)'%(Gf,Pf[0]))
    hist=[Gf]
    for it in range(iters):
        r1,r2=rng.rand(n_part,dim),rng.rand(n_part,dim)
        V=np.clip(w*V+c1*r1*(P-X)+c2*r2*(G-X),-0.25,0.25)
        X=np.clip(X+V,0.0,1.0)
        f=np.array([fitness(x) for x in X])
        imp=f<Pf; P[imp]=X[imp]; Pf[imp]=f[imp]
        if Pf.min()<Gf: g=int(np.argmin(Pf)); G=P[g].copy(); Gf=float(Pf.min())
        hist.append(Gf); print('  PSO iter %d: best %.3f'%(it+1,Gf))
    return G,Gf,hist

t0=time.time()
PSO_VEC,PSO_OBJ,PSO_HIST=pso(lambda v: objective(decode(v)),len(SPACE),x0=BASE_VEC)
PSO_KW=decode(PSO_VEC)
print('PSO best objective %.3f in %.0f s'%(PSO_OBJ,time.time()-t0))

# ---------------- matched head-to-head before adopting anything -------------
print('\nre-scoring all three candidates on IDENTICAL seeds and budget (%d seeds, %d epochs):'
      %(COMPARE_SEEDS,EPOCHS))
cands=[]
for nm,kw in (('baseline',{}),('Bayesian',BO_KW),('PSO',PSO_KW)):
    o=objective(kw,n_seeds=COMPARE_SEEDS)
    cands.append((nm,o,kw)); print('   %-10s %.3f'%(nm,o))
SRC,BEST_OBJ,FINAL_KW=min(cands,key=lambda x:x[1])
print('\n-> adopting: %s (%.3f)'%(SRC,BEST_OBJ))
if FINAL_KW:
    for k,v in sorted(FINAL_KW.items()): print('   %-11s %s'%(k,v))

  PSO init best 8.131 (particle 0 = baseline: 8.131)
  PSO iter 1: best 8.057
  PSO iter 2: best 8.057
  PSO iter 3: best 8.057
  PSO iter 4: best 8.057
PSO best objective 8.057 in 6527 s

re-scoring all three candidates on IDENTICAL seeds and budget (3 seeds, 60 epochs):
   baseline   8.285
   Bayesian   8.309
   PSO        8.491

-> adopting: baseline (8.285)


In [8]:
vs,ts,prm=[],[],None
for s in range(SEEDS_FINAL):
    v,t,prm=run(BA,BP,SEED+1000+s,epochs=EPOCHS,**FINAL_KW); vs.append(v); ts.append(t)
FINAL=dict(arch=BA,prep=BP,source=SRC,test_mean=float(np.mean(ts)),
           test_std=float(np.std(ts,ddof=1)),test_best=float(np.min(ts)),
           params=int(prm),per_seed=[round(x,3) for x in ts],kw={k:float(v) for k,v in FINAL_KW.items()})
base=float(G[(G.arch==BA)&(G.prep==BP)]['test_mean'].iloc[0])
print('='*68)
print('%-30s %.2f'%('baseline (%s+%s)'%(BA,BP),base))
print('%-30s %.2f +/- %.2f (best run %.2f)'%('final via %s'%SRC,FINAL['test_mean'],
      FINAL['test_std'],FINAL['test_best']))
print('%-30s %.2f'%('Peng published test value',9.99))
print('='*68)

baseline (ours+peng)           11.12
final via baseline             10.84 +/- 0.28 (best run 10.55)
Peng published test value      9.99


## Part D — corrected benchmark table

In [9]:
PUB=[('1-FCLCNN-LSTM','Peng et al. 2021 (Table 6, test)',9.99),
     ('LSTM with attention','Muneer et al.',11.23),
     ('HDNN','Al-Dulaimi et al. 2019',12.22),('DCNN','Li, Ding & Sun 2018',12.64),
     ('Attention-based DCNN','Muneer et al.',13.08),('MSTSDN','Liu et al. 2024',13.66),
     ('LSTMBS','Liao et al.',16.00),('Rulclipper','Ramasso',16.00),
     ('Deep LSTM','Zheng et al. 2017',16.18),('GB','Zhang et al.',16.84),
     ('FADCNN','Li et al.',19.82),('Random Forest','Zheng et al.',20.27)]
rows=[dict(method=m,source=s,rmse=v) for m,s,v in PUB]
for r in grid:
    rows.append(dict(method='Ours: %s arch + %s prep (5-seed mean)'%(r['arch'],r['prep']),
                     source='this work',rmse=round(r['test_mean'],2)))
rows.append(dict(method='Ours: final (%s, %s+%s)'%(SRC,BA,BP),source='this work',
                 rmse=round(FINAL['test_mean'],2)))
rows.append(dict(method='   our best single run',source='this work',rmse=round(FINAL['test_best'],2)))
rows.append(dict(method='   trivial: predict training mean',source='reference',rmse=round(TRIVIAL,2)))
B=pd.DataFrame(rows).sort_values('rmse').reset_index(drop=True); B.index=B.index+1
print(B.to_string())
B.to_csv('benchmark_fd003_v3.csv'); G.to_csv('factor_grid_fd003.csv',index=False)
json.dump(dict(grid=grid,baseline_obj=BASE_OBJ,bo_obj=float(BO_OBJ),pso_obj=float(PSO_OBJ),
               pso_history=[float(x) for x in PSO_HIST],adopted=SRC,final=FINAL,
               peng_published_test=9.99,peng_published_train=7.56,trivial=TRIVIAL),
          open('fd003_results_v3.json','w'),indent=2,default=str)
print('\nsaved benchmark_fd003_v3.csv, factor_grid_fd003.csv, fd003_results_v3.json')

                                             method                            source   rmse
1                                     1-FCLCNN-LSTM  Peng et al. 2021 (Table 6, test)   9.99
2                               our best single run                         this work  10.55
3                 Ours: final (baseline, ours+peng)                         this work  10.84
4         Ours: ours arch + peng prep (5-seed mean)                         this work  11.12
5                               LSTM with attention                     Muneer et al.  11.23
6                                              HDNN            Al-Dulaimi et al. 2019  12.22
7                                              DCNN               Li, Ding & Sun 2018  12.64
8         Ours: ours arch + ours prep (5-seed mean)                         this work  13.05
9   Ours: ours arch + peng_nocyc prep (5-seed mean)                         this work  13.08
10                             Attention-based DCNN                   